# Explanation

The following does a Fourier fit of a torsional potential, matching results from MESS.

In [1]:
from learn_vrc_tst import View, geom, coord

geo = geom.read_xyz_file("geom.xyz")
geo = geom.translate(geo, -geom.center_of_mass(geo))

In [2]:
from collections.abc import Collection
import pint
import numpy as np
from scipy import linalg

def raw_torsional_mode(geo: geom.Geometry, axis: tuple[int, int], group: Collection[int]) -> np.ndarray:
    group = sorted(set(group))
    ax_idx1, ax_idx2 = axis
    ax1 = geo.coordinates[ax_idx1]
    ax2 = geo.coordinates[ax_idx2]
    ax = ax2 - ax1
    ax /= linalg.norm(ax)
    mode = np.zeros_like(geo.coordinates)
    mode[group] = np.cross(ax, geo.coordinates[group] - ax2)
    return mode


def mode_translational_component(geo: geom.Geometry, mode: np.ndarray) -> np.ndarray:
    M = np.sum(geo.masses)
    m = np.asarray(geo.masses)
    return np.sum(m[:, np.newaxis] * mode, axis=0) / M


def mode_rotational_component(geo: geom.Geometry, mode: np.ndarray) -> np.ndarray:
    I = geom.inertia_tensor(geo)
    I_inv = linalg.inv(I, assume_a="sym")
    m = np.asarray(geo.masses)
    omega = I_inv @ np.sum(m[:, np.newaxis] * np.cross(geo.coordinates, mode), axis=0)
    return np.cross(omega, geo.coordinates)

def internal_mode_component(geo: geom.Geometry, mode: np.ndarray) -> np.ndarray:
    mode_trans = mode_translational_component(geo, mode)
    mode_rot = mode_rotational_component(geo, mode)
    return mode - mode_trans - mode_rot


def mode_effective_mass(geo: geom.Geometry, mode: np.ndarray) -> float:
    return np.sum(geo.masses * np.linalg.norm(mode, axis=1) ** 2)

mode = raw_torsional_mode(geo, (7, 11), [14])
mode = internal_mode_component(geo, mode)
mu = mode_effective_mass(geo, mode)
mu = pint.Quantity(mu, "dalton angstrom**2")
h = pint.Quantity(1, "planck_constant")
hbar = h / (2 * np.pi)
prefactor = (hbar ** 2 / (2 * mu))

c = pint.Quantity(1, "speed_of_light")
const = (prefactor / (h * c)).m_as("cm**-1")
print(f"{const = }")

view = View()
view.add_geometry(geo, label=True)
for coord, motion in zip(geo.coordinates, mode):
    if np.linalg.norm(motion) > 1e-3:
        view.add_arrow(motion + coord, coord)
view.show()

const = np.float64(20.771663236202357)


3Dmol.js failed to load for some reason. Please check your browser console for error messages.